# D1 Refinement — Hyperparameter Sweep + Multi-Seed Runs

Responds to advisor feedback at Standup I: *"patience=2 is not much, and parameters
can be changed for really different results."*

**Two stages**
1. **Grid search** over learning rate, one seed, patience raised to 4.
   Best configuration chosen by **dev** macro F1 — the test set is never used for selection.
2. **Multi-seed** runs at each model's best configuration, so results are reported as
   **mean ± std** instead of a single number.

Every model gets the *same* search budget and the *same* dev-only selection rule, so the
comparison stays fair while each model is shown at its own best setting.

**Fully resumable** — results append to `sweep_results.json` in Drive. If Colab
disconnects, just re-run; completed runs are skipped.

In [ ]:
# ============================================================
# 0. Environment
# ============================================================
!pip -q install "transformers>=4.40" "datasets>=2.19" accelerate evaluate

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
if not torch.cuda.is_available():
    raise SystemExit("No GPU! Runtime -> Change runtime type -> T4 GPU, then rerun.")

In [ ]:
# ============================================================
# 1. Sweep configuration
# ============================================================
# Start with Tamil only (the scientifically interesting case, and fastest).
# Add "malayalam" / "english" later if time allows.
LANGS   = ["tamil"]
MODELS  = ["xlm-roberta-base", "google/muril-base-cased"]

LR_GRID      = [1e-5, 2e-5, 3e-5]   # stage 1 searches over these
EPOCHS       = 8
PATIENCE     = 5                    # was 2 - advisor feedback
GRID_SEED    = 42                   # single seed for the grid
EXTRA_SEEDS  = [1234, 2026]         # stage 2 adds these at the best config

MAX_LEN, BATCH_SIZE, WEIGHT_DECAY = 128, 16, 0.01

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json, gc, shutil, time

BASE_DIR    = Path("/content/drive/MyDrive/hope_speech")
DATA_DIR    = BASE_DIR / "processed"
METRICS_OUT = BASE_DIR / "metrics"
METRICS_OUT.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = METRICS_OUT / "sweep_results.json"

def load_results():
    if RESULTS_PATH.exists():
        return json.loads(RESULTS_PATH.read_text())
    return {"runs": []}

def save_results(res):
    RESULTS_PATH.write_text(json.dumps(res, indent=2))

def already_done(res, model, lang, lr, seed):
    return any(r["model"] == model and r["lang"] == lang
               and abs(r["lr"] - lr) < 1e-12 and r["seed"] == seed
               for r in res["runs"])

results = load_results()
print(f"Config OK - {len(results['runs'])} runs already recorded")
total = len(MODELS) * len(LANGS) * (len(LR_GRID) + len(EXTRA_SEEDS))
print(f"Planned total runs: {total}")

In [ ]:
# ============================================================
# 2. Data loading
# ============================================================
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer

def load_split(lang, split):
    df = pd.read_csv(DATA_DIR / f"{lang}_{split}_processed.csv")
    df["text"] = df["text"].fillna("").astype(str)
    df = df[df["text"].str.strip() != ""]
    assert set(df["label"].unique()) <= {0, 1}
    return df[["text", "label"]].reset_index(drop=True)

_cache = {}
def get_data(lang, model_name):
    """Tokenized datasets are tokenizer-specific, so cache per (lang, model)."""
    key = (lang, model_name)
    if key in _cache:
        return _cache[key]
    tok = AutoTokenizer.from_pretrained(model_name)
    dfs = {s: load_split(lang, s) for s in ("train", "dev", "test")}
    ds = {}
    for name, df in dfs.items():
        d = Dataset.from_pandas(df, preserve_index=False)
        ds[name] = d.map(lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN),
                         batched=True, remove_columns=["text"])
    _cache[key] = (ds, dfs, tok)
    return _cache[key]

In [ ]:
# ============================================================
# 3. One training run
# ============================================================
import numpy as np
from transformers import (AutoModelForSequenceClassification, Trainer, TrainingArguments,
                          DataCollatorWithPadding, EarlyStoppingCallback, set_seed)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro"),
            "hope_f1": f1_score(labels, preds, pos_label=1),
            "hope_precision": precision_score(labels, preds, pos_label=1, zero_division=0),
            "hope_recall": recall_score(labels, preds, pos_label=1, zero_division=0)}

def train_one(model_name, lang, lr, seed, stage):
    """Train one configuration; return a result dict (dev-selected, test-evaluated once)."""
    t0 = time.time()
    ds, dfs, tok = get_data(lang, model_name)
    set_seed(seed)

    cw = compute_class_weight("balanced", classes=np.array([0, 1]),
                              y=dfs["train"]["label"].values)
    cw_t = torch.tensor(cw, dtype=torch.float32).cuda()

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=cw_t)
            return (loss, outputs) if return_outputs else loss

    ckpt = Path("/content/ckpt_sweep")
    shutil.rmtree(ckpt, ignore_errors=True)
    total_steps = (len(ds["train"]) / BATCH_SIZE) * EPOCHS

    args = TrainingArguments(
        output_dir=str(ckpt),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=64,
        learning_rate=lr,
        warmup_steps=int(0.1 * total_steps),
        weight_decay=WEIGHT_DECAY,
        fp16=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=1,
        logging_steps=500,
        seed=seed,
        report_to="none",
        disable_tqdm=True,
    )

    trainer = WeightedTrainer(
        model=model, args=args,
        train_dataset=ds["train"], eval_dataset=ds["dev"],
        data_collator=DataCollatorWithPadding(tok),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    )
    trainer.train()

    # per-epoch dev curve -> lets us check convergence and spot degenerate runs
    curve = [round(h["eval_macro_f1"], 4)
             for h in trainer.state.log_history if "eval_macro_f1" in h]
    train_losses = [round(h["loss"], 4)
                    for h in trainer.state.log_history if "loss" in h]

    dev_out  = trainer.evaluate(ds["dev"])
    test_out = trainer.predict(ds["test"]).metrics

    rec = {
        "stage": stage, "model": model_name, "lang": lang,
        "lr": lr, "epochs": EPOCHS, "patience": PATIENCE, "seed": seed,
        "dev_macro_f1":  round(dev_out["eval_macro_f1"], 4),
        "dev_hope_f1":   round(dev_out["eval_hope_f1"], 4),
        "test_macro_f1": round(test_out["test_macro_f1"], 4),
        "test_hope_f1":  round(test_out["test_hope_f1"], 4),
        "test_accuracy": round(test_out["test_accuracy"], 4),
        "best_dev_metric": round(trainer.state.best_metric, 4),
        "dev_curve": curve,
        "degenerate": max(curve) < 0.45,     # flag collapsed runs
        "minutes": round((time.time() - t0) / 60, 1),
    }

    del model, trainer
    gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(ckpt, ignore_errors=True)
    return rec

def short(m):
    return "XLM-R" if "xlm" in m else "MuRIL"

In [ ]:
# ============================================================
# 4. STAGE 1 - learning-rate grid (dev-selected)
# ============================================================
for lang in LANGS:
    for model_name in MODELS:
        for lr in LR_GRID:
            if already_done(results, model_name, lang, lr, GRID_SEED):
                print(f"skip  {short(model_name):6s} {lang:10s} lr={lr:.0e} seed={GRID_SEED}")
                continue
            print(f"RUN   {short(model_name):6s} {lang:10s} lr={lr:.0e} seed={GRID_SEED} ...")
            rec = train_one(model_name, lang, lr, GRID_SEED, stage="grid")
            results["runs"].append(rec)
            save_results(results)
            flag = "  <-- DEGENERATE" if rec["degenerate"] else ""
            print(f"      dev {rec['dev_macro_f1']:.4f} | test {rec['test_macro_f1']:.4f}"
                  f" | {rec['minutes']}min{flag}")
print("\nStage 1 complete.")

In [ ]:
# ============================================================
# 5. STAGE 2 - multi-seed at each model's best config
# ============================================================
def best_lr(res, model_name, lang):
    grid = [r for r in res["runs"] if r["stage"] == "grid"
            and r["model"] == model_name and r["lang"] == lang and not r["degenerate"]]
    if not grid:
        return None
    return max(grid, key=lambda r: r["dev_macro_f1"])["lr"]

for lang in LANGS:
    for model_name in MODELS:
        lr = best_lr(results, model_name, lang)
        if lr is None:
            print(f"!! no usable grid run for {short(model_name)} {lang}; skipping seeds")
            continue
        print(f"\n{short(model_name)} {lang}: best dev lr = {lr:.0e}")
        for seed in EXTRA_SEEDS:
            if already_done(results, model_name, lang, lr, seed):
                print(f"skip  seed={seed}")
                continue
            print(f"RUN   {short(model_name):6s} {lang:10s} lr={lr:.0e} seed={seed} ...")
            rec = train_one(model_name, lang, lr, seed, stage="seed")
            results["runs"].append(rec)
            save_results(results)
            flag = "  <-- DEGENERATE" if rec["degenerate"] else ""
            print(f"      dev {rec['dev_macro_f1']:.4f} | test {rec['test_macro_f1']:.4f}"
                  f" | {rec['minutes']}min{flag}")
print("\nStage 2 complete.")

In [ ]:
# ============================================================
# 6. Summary - the tables that go straight onto the slides
# ============================================================
import statistics as st

res = load_results()
print("=" * 72)
print("STAGE 1 - LEARNING-RATE GRID (selected on DEV)")
print("=" * 72)
for lang in LANGS:
    for model_name in MODELS:
        rows = sorted([r for r in res["runs"] if r["stage"] == "grid"
                       and r["model"] == model_name and r["lang"] == lang],
                      key=lambda r: -r["dev_macro_f1"])
        if not rows: continue
        print(f"\n{short(model_name)} / {lang}")
        print(f"  {'lr':>8} {'dev macro F1':>13} {'test macro F1':>14}  note")
        for r in rows:
            note = "DEGENERATE" if r["degenerate"] else ("<- best dev" if r is rows[0] else "")
            print(f"  {r['lr']:>8.0e} {r['dev_macro_f1']:>13.4f} {r['test_macro_f1']:>14.4f}  {note}")

print("\n" + "=" * 72)
print("STAGE 2 - BEST CONFIG, MULTIPLE SEEDS (test macro F1)")
print("=" * 72)
summary = {}
for lang in LANGS:
    for model_name in MODELS:
        lr = best_lr(res, model_name, lang)
        if lr is None: continue
        runs = [r for r in res["runs"] if r["model"] == model_name and r["lang"] == lang
                and abs(r["lr"] - lr) < 1e-12 and not r["degenerate"]]
        vals = [r["test_macro_f1"] for r in runs]
        hope = [r["test_hope_f1"] for r in runs]
        if not vals: continue
        mean = st.mean(vals)
        sd   = st.stdev(vals) if len(vals) > 1 else 0.0
        summary[f"{short(model_name)}|{lang}"] = {
            "lr": lr, "n_seeds": len(vals),
            "test_macro_f1_mean": round(mean, 4), "test_macro_f1_std": round(sd, 4),
            "test_macro_f1_runs": vals,
            "test_hope_f1_mean": round(st.mean(hope), 4),
        }
        print(f"\n{short(model_name)} / {lang}  (lr={lr:.0e}, n={len(vals)})")
        print(f"  runs : {', '.join(f'{v:.4f}' for v in vals)}")
        print(f"  mean : {mean:.4f}  +/- {sd:.4f}")

(METRICS_OUT / "sweep_summary.json").write_text(json.dumps(summary, indent=2))
print("\nSaved -> metrics/sweep_results.json and metrics/sweep_summary.json")
print(f"Total runs recorded: {len(res['runs'])}")